# TabPFN

A transformer pretrained on millions of synthetic tables. It is not trained on our data: it reads the training rows and predicts the new rows in one forward pass, so there are no hyperparameters to tune.

**First run:** the pretrained weights are downloaded once. A browser page opens to log in to Prior Labs and accept the model license; the acceptance is cached afterwards.

In [3]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from tabpfn import TabPFNRegressor

X = pd.read_csv("../X_train_log_knn.csv", index_col="id")
X_test = pd.read_csv("../X_test_log_knn.csv", index_col="id")
y = pd.read_csv("../y_train.csv", index_col="id").loc[X.index, "y"]
fold = pd.read_csv("../Stacking/folds.csv", index_col="id").loc[X.index, "fold"]
print(f"train {X.shape} | test {X_test.shape} | y {y.shape}")

train (1152, 216) | test (776, 216) | y (1152,)


In [5]:
# Out-of-fold predictions on the shared folds (runs on the M4 GPU through MPS)

oof = pd.Series(np.nan, index=X.index, name="tabpfn")
for f in range(5):
    train, val = fold != f, fold == f
    start = time.time()
    model = TabPFNRegressor(random_state=0).fit(X[train], y[train])
    oof[val] = model.predict(X[val])
    print(f"fold {f}: R2 {r2_score(y[val], oof[val]):.4f} ({time.time() - start:.0f} s)")

print(f"out-of-fold R2: {r2_score(y, oof):.4f}")
oof.to_csv("../Stacking/oof_tabpfn.csv")

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

fold 0: R2 0.8074 (90 s)
fold 1: R2 0.7970 (10 s)
fold 2: R2 0.7267 (11 s)
fold 3: R2 0.7669 (10 s)
fold 4: R2 0.7828 (11 s)
out-of-fold R2: 0.7772


In [6]:
# Compare with the models already in the stack: their score, and how similar their errors are to TabPFN's
# (close to 1 = same errors, so nothing new for the stack)

others = pd.concat({m: pd.read_csv(f"../Stacking/oof_{m}.csv", index_col="id").iloc[:, 0]
                    for m in ["svr", "xgb", "gp"]}, axis=1).loc[X.index]
for m in others:
    print(f"{m:4s}: out-of-fold R2 {r2_score(y, others[m]):.4f} | "
          f"correlation of its errors with TabPFN's: {(others[m] - y).corr(oof - y):.3f}")

svr : out-of-fold R2 0.7083 | correlation of its errors with TabPFN's: 0.875
xgb : out-of-fold R2 0.6787 | correlation of its errors with TabPFN's: 0.834
gp  : out-of-fold R2 0.6896 | correlation of its errors with TabPFN's: 0.779


In [7]:
# Full model: all 1152 training rows, predicting the 776 test rows (sample.csv format, for Stacking/stacking.ipynb)

model = TabPFNRegressor(random_state=0).fit(X, y)
pd.Series(model.predict(X_test), index=X_test.index, name="y").to_csv("submission_tabpfn.csv")